### **Import Library**

In [31]:
import numpy as np
import pandas as pd

import sys
sys.path.append('.')
from data import supply, demand, cost, agents, destinations

In [32]:
allocation_lc = np.load('allocation_lc.npy')
print(f"  Z Least Cost = Rp. {np.sum(allocation_lc * cost_np):,.1f} ribu")

  Z Least Cost = Rp. 52,385.0 ribu


In [33]:
df = pd.DataFrame(allocation_lc, index=short_ag, columns=short_dest)
df.index.name = 'Dari/Ke'
df['Supply'] = supply
df.loc['Demand'] = demand + [sum(supply)]
df.map(lambda x: '-' if x == 0.0 else (int(x) if float(x).is_integer() else x))

,CS,KD,BN,PG,KT,CB,CSK,PB,CH,CJ,Supply
Dari/Ke,,,,,,,,,,,
A1,300,-,200,-,-,50,-,-,-,50,600
A2,-,-,-,300,-,-,50,-,100,-,450
A3,-,100,-,-,50,50,-,50,-,-,250
Demand,300,100,200,300,50,100,50,50,100,50,1300


### **Algoritma MODI**

In [34]:
def compute_uv(allocation, cost):
    m, n = allocation.shape
    u = [None] * m
    v = [None] * n
    u[0] = 0 
    changed = True
    while changed:
        changed = False
        for i in range(m):
            for j in range(n):
                if allocation[i][j] > 0:
                    if u[i] is not None and v[j] is None:
                        v[j] = cost[i][j] - u[i]
                        changed = True
                    elif v[j] is not None and u[i] is None:
                        u[i] = cost[i][j] - v[j]
                        changed = True
    return u, v

In [35]:
def compute_opportunity_cost(allocation, cost, u, v):
    m, n = allocation.shape
    d = np.full((m, n), np.nan)
    for i in range(m):
        for j in range(n):
            if allocation[i][j] == 0 and u[i] is not None and v[j] is not None:
                d[i][j] = cost[i][j] - u[i] - v[j]
    return d

In [36]:
def find_loop(allocation, enter_i, enter_j):
    m, n = allocation.shape
    
    def search(path):
        i, j = path[-1]
        if len(path) % 2 == 1:
            for jj in range(n):
                if jj == j: continue
                if allocation[i][jj] > 0 or (i == enter_i and jj == enter_j):
                    if len(path) >= 4 and i == enter_i and jj == enter_j:
                        return path
                    if (i, jj) not in path:
                        result = search(path + [(i, jj)])
                        if result: return result
        else: 
            for ii in range(m):
                if ii == i: continue
                if allocation[ii][j] > 0 or (ii == enter_i and j == enter_j):
                    if len(path) >= 4 and ii == enter_i and j == enter_j:
                        return path
                    if (ii, j) not in path:
                        result = search(path + [(ii, j)])
                        if result: return result
        return None
    
    return search([(enter_i, enter_j)])

In [37]:
def modi_method(allocation_init, cost):
    allocation = allocation_init.copy()
    cost_np = np.array(cost)
    iterations = []
    iteration = 1
    
    while True:
        u, v = compute_uv(allocation, cost)
        d = compute_opportunity_cost(allocation, cost_np, u, v)
        min_d = np.nanmin(d)
        
        log = {
            'Iterasi': iteration,
            'u': u.copy(),
            'v': v.copy(),
            'd': d.copy(),
            'min_d': min_d,
            'allocation': allocation.copy(),
            'Z': np.sum(allocation * cost_np)
        }
        
        if min_d >= 0:
            log['status'] = 'OPTIMAL'
            iterations.append(log)
            break
        
        enter_pos = np.unravel_index(np.nanargmin(d), d.shape)
        enter_i, enter_j = enter_pos
        log['enter_cell'] = (enter_i, enter_j)
        
        loop = find_loop(allocation, enter_i, enter_j)
        log['loop'] = loop
        log['status'] = f'Realokasi → sel ({enter_i},{enter_j})'
        iterations.append(log)
        
        if loop is None:
            print(f"Loop tidak ditemukan di iterasi {iteration}")
            break
        
        minus_cells = loop[1::2]
        theta = min(allocation[i][j] for i, j in minus_cells)
        
        for idx, (i, j) in enumerate(loop):
            if idx % 2 == 0:
                allocation[i][j] += theta  
            else:
                allocation[i][j] -= theta 
        
        iteration += 1
    
    return allocation, iterations

In [38]:
allocation_optimal, iterations = modi_method(allocation_lc, cost)

In [39]:
print(f"Jumlah iterasi MODI : {len(iterations)}")
print(f"Status akhir        : {iterations[-1]['status']}")
print(f"Z awal (Least Cost) : Rp. {np.sum(allocation_lc * cost_np):,.1f} ribu")
print(f"Z optimal (MODI)    : Rp. {np.sum(allocation_optimal * cost_np):,.1f} ribu")

Jumlah iterasi MODI : 2
Status akhir        : OPTIMAL
Z awal (Least Cost) : Rp. 52,385.0 ribu
Z optimal (MODI)    : Rp. 52,270.0 ribu


### **Hasil MODI**

In [40]:
df_opt = pd.DataFrame(allocation_optimal, index=short_ag, columns=short_dest)
df_opt.index.name = 'Dari/Ke'
df_opt['Supply'] = supply
df_opt.loc['Demand'] = demand + [sum(supply)]
df_opt.map(lambda x: '-' if x == 0.0 else (int(x) if float(x).is_integer() else x))


,CS,KD,BN,PG,KT,CB,CSK,PB,CH,CJ,Supply
Dari/Ke,,,,,,,,,,,
A1,250,-,200,-,-,100,-,-,-,50,600
A2,-,-,-,300,-,-,50,-,100,-,450
A3,50,100,-,-,50,-,-,50,-,-,250
Demand,300,100,200,300,50,100,50,50,100,50,1300


In [41]:
rincian = []
for i in range(len(supply)):
    for j in range(len(demand)):
        if allocation_optimal[i][j] > 0:
            subtotal = allocation_optimal[i][j] * cost[i][j]
            rincian.append({
                'Dari'               : agents[i].split('(')[0].strip(),
                'Ke'                 : destinations[j],
                'Alokasi'            : int(allocation_optimal[i][j]),
                'Biaya/unit (Rp.rb)' : cost[i][j],
                'Subtotal (Rp.rb)'   : round(subtotal, 1)
            })

In [42]:
df_rincian = pd.DataFrame(rincian)
print(df_rincian.to_string(index=False))

  Dari                   Ke  Alokasi  Biaya/unit (Rp.rb)  Subtotal (Rp.rb)
Agen 1 Curug Sangereng (CS)      250                39.1            9775.0
Agen 1   Bojong Nangka (BN)      200                37.6            7520.0
Agen 1          Cibogo (CB)      100                40.0            4000.0
Agen 1        Cijantra (CJ)       50                37.6            1880.0
Agen 2      Pagedangan (PG)      300                42.1           12630.0
Agen 2         Cisauk (CSK)       50                41.1            2055.0
Agen 2          Cihuni (CH)      100                40.1            4010.0
Agen 3 Curug Sangereng (CS)       50                42.1            2105.0
Agen 3      Kelapa Dua (KD)      100                40.6            4060.0
Agen 3   Karang Tengah (KT)       50                43.3            2165.0
Agen 3 Pakulonan Barat (PB)       50                41.4            2070.0


In [43]:
total_optimal = np.sum(allocation_optimal * cost_np)
print(f"  Total Biaya Optimal (Z) = Rp. {total_optimal:,.1f} ribu")
print(f"                          = Rp. {total_optimal * 1000:,.0f}")

  Total Biaya Optimal (Z) = Rp. 52,270.0 ribu
                          = Rp. 52,270,000


### **Perbandingan Sebelum vs Sesudah Optimasi**

In [44]:
z_lc      = np.sum(allocation_lc * cost_np)
z_optimal = np.sum(allocation_optimal * cost_np)
selisih   = z_lc - z_optimal
persen    = (selisih / z_lc) * 100

df_compare = pd.DataFrame({
    'Metode'             : ['Least Cost (Solusi Awal)', 'MODI (Solusi Optimal)'],
    'Total Biaya (Rp.rb)': [f'{z_lc:,.1f}', f'{z_optimal:,.1f}'],
    'Total Biaya (Rp)'   : [f'Rp. {z_lc*1000:,.0f}', f'Rp. {z_optimal*1000:,.0f}']
})

In [45]:
print(df_compare.to_string(index=False))
print(f"\nPenghematan : Rp. {selisih:,.1f} ribu = Rp. {selisih*1000:,.0f}")
print(f"Efisiensi   : {persen:.2f}%")

                  Metode Total Biaya (Rp.rb) Total Biaya (Rp)
Least Cost (Solusi Awal)            52,385.0   Rp. 52,385,000
   MODI (Solusi Optimal)            52,270.0   Rp. 52,270,000

Penghematan : Rp. 115.0 ribu = Rp. 115,000
Efisiensi   : 0.22%
